In [1]:
import pandas as pd 
import numpy as np

In [2]:
def to_count(s):
    return pd.to_numeric(s.replace('-', 0), errors='raise').astype(int)

In [3]:
registrations_by_brands_sep2026 = pd.read_excel('../data/raw/fz28_2026_09.xlsx', header=None, sheet_name="FZ 28.4")
registrations_by_brands_ytd2526 = pd.read_excel('../data/raw/fz28_2026_09.xlsx', header=None, sheet_name="FZ 28.5")


brands_sep2026 = registrations_by_brands_sep2026.iloc[12:79, 1].str.lower().str.strip()

total_registrations_sep2026 = to_count(registrations_by_brands_sep2026.iloc[12:79, 2])
electric_drive_registrations_sep2026 = to_count(registrations_by_brands_sep2026.iloc[12:79,5])
bev_registrations_sep2026 = to_count(registrations_by_brands_sep2026.iloc[12:79, 7])
phev_registrations_sep2026 = to_count(registrations_by_brands_sep2026.iloc[12:79, 9])


brands_jan_sep = registrations_by_brands_ytd2526.iloc[12:82, 1].str.lower().str.strip()

total_registrations_ytd26 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 2])
electric_drive_registrations_ytd26 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 5])
bev_registrations_ytd26 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 7])
phev_registrations_ytd26 = electric_drive_registrations_ytd26 - bev_registrations_ytd26

total_registrations_ytd25 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 9])
electric_drive_registrations_ytd25 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 12])
bev_registrations_ytd25 = to_count(registrations_by_brands_ytd2526.iloc[12:82, 14])
phev_registrations_ytd25 = electric_drive_registrations_ytd25 - bev_registrations_ytd25


assert (electric_drive_registrations_sep2026 - bev_registrations_sep2026 == phev_registrations_sep2026).all()
assert (bev_registrations_sep2026 + phev_registrations_sep2026 <= total_registrations_sep2026).all()
assert (bev_registrations_ytd26 + phev_registrations_ytd26 <= total_registrations_ytd26).all()
assert (bev_registrations_ytd25 + phev_registrations_ytd25 <= total_registrations_ytd25).all()

registrations_by_brands_sep2026_filtered = pd.DataFrame({
    'brand': brands_sep2026,    
    'total_sep26': total_registrations_sep2026,
    'bev_sep26': bev_registrations_sep2026,
    'phev_sep26': phev_registrations_sep2026
}).reset_index(drop=True)



registrations_by_brands_jan_sep_filtered = pd.DataFrame({
    'brand': brands_jan_sep,
    'total_ytd26': total_registrations_ytd26,
    'bev_ytd26': bev_registrations_ytd26,
    'phev_ytd26': phev_registrations_ytd26 ,
    'total_ytd25' : total_registrations_ytd25,
    'bev_ytd25': bev_registrations_ytd25,
    'phev_ytd25': phev_registrations_ytd25,
    
}).reset_index(drop=True)


sep_cols = ['total_sep26', 'bev_sep26', 'phev_sep26']

final_df = registrations_by_brands_sep2026_filtered.merge(
    registrations_by_brands_jan_sep_filtered,
    how='outer',
    indicator=True,
    on='brand',
    validate='one_to_one'
)


final_df[sep_cols] = final_df[sep_cols].fillna(0).astype(int)



assert (final_df['_merge'] != 'left_only').all()
# print(final_df.loc[final_df['_merge'] == 'right_only', 'brand'])

final_df = final_df.drop(columns='_merge')

final_df = final_df.sort_values('brand').reset_index(drop=True)



assert registrations_by_brands_sep2026.iloc[79, 1].strip().lower() == 'insgesamt'
assert registrations_by_brands_sep2026.iloc[79,2] == total_registrations_sep2026.sum()
assert registrations_by_brands_sep2026.iloc[79,5] == electric_drive_registrations_sep2026.sum()
assert registrations_by_brands_sep2026.iloc[79,7] == bev_registrations_sep2026.sum()
assert registrations_by_brands_sep2026.iloc[79,9] == phev_registrations_sep2026.sum()


assert registrations_by_brands_ytd2526.iloc[82, 1].strip().lower() == 'insgesamt'
assert registrations_by_brands_ytd2526.iloc[82,2] == total_registrations_ytd26.sum()
assert registrations_by_brands_ytd2526.iloc[82,5] == electric_drive_registrations_ytd26.sum()
assert registrations_by_brands_ytd2526.iloc[82,7] == bev_registrations_ytd26.sum()


assert registrations_by_brands_ytd2526.iloc[82,9] == total_registrations_ytd25.sum()
assert registrations_by_brands_ytd2526.iloc[82,12] == electric_drive_registrations_ytd25.sum()
assert registrations_by_brands_ytd2526.iloc[82,14] == bev_registrations_ytd25.sum()


market = final_df[['total_ytd26', 'total_ytd25', 'bev_ytd26', 'bev_ytd25',
                      'phev_ytd26', 'phev_ytd25']].sum()

market_growth = {t: (market[f'{t}_ytd26'] - market[f'{t}_ytd25']) / market[f'{t}_ytd25'] * 100
                 for t in ['total', 'bev', 'phev']}
print(market_growth)

for t in ['total', 'bev', 'phev']:
    final_df[f'{t}_growth_ytd'] = (
        (final_df[f'{t}_ytd26'] - final_df[f'{t}_ytd25'])
        / final_df[f'{t}_ytd25'].replace(0, np.nan)
        * 100
    )

periods = ['sep26', 'ytd26', 'ytd25']

for p in periods:
    final_df[f'market_share_total_{p}'] = final_df[f'total_{p}'] / final_df[f'total_{p}'].sum() * 100
    for t in ['bev', 'phev']:
        final_df[f'{t}_market_share_{p}'] = final_df[f'{t}_{p}'] / final_df[f'{t}_{p}'].sum() * 100
        final_df[f'internal_{t}_share_{p}'] = final_df[f'{t}_{p}'] / final_df[f'total_{p}'] * 100


metrics = ['market_share_total', 'bev_market_share', 'internal_bev_share',
              'phev_market_share', 'internal_phev_share']

for m in metrics:
       final_df[f'{m}_change_pp'] = final_df[f'{m}_ytd26'] - final_df[f'{m}_ytd25']

cols = [f'{m}_change_pp' for m in metrics]

growth_cols = ['brand','total_growth_ytd', 'bev_growth_ytd', 'phev_growth_ytd']
# display(
#     final_df[final_df['brand'].isin(['bmw', 'vw', 'audi', 'volvo', 'mercedes'])]
#     [growth_cols + cols].round(2)
#     )


outperformers = final_df[
    (final_df['total_growth_ytd'] >= market_growth['total'])
    & (final_df['bev_growth_ytd'] >= market_growth['bev'])
    & (final_df['phev_growth_ytd'] >= market_growth['phev'])
].sort_values('total_growth_ytd', ascending=False)

display(outperformers[['brand', 'total_ytd26'] + growth_cols[1:] + cols].round(2))

{'total': np.float64(5.281119512042563), 'bev': np.float64(58.06929320097749), 'phev': np.float64(13.63271152901422)}


,brand,total_ytd26,total_growth_ytd,bev_growth_ytd,phev_growth_ytd,market_share_total_change_pp,bev_market_share_change_pp,internal_bev_share_change_pp,phev_market_share_change_pp,internal_phev_share_change_pp
6,byd,42788,262.06,128.87,465.87,1.37,0.84,-22.24,8.54,22.25
33,leapmotor,16044,261.27,295.40,48.78,0.51,1.50,8.14,0.09,-8.15
37,lynk & co,927,59.28,270.59,46.17,0.01,0.01,7.75,0.07,-7.75
57,skoda,186973,12.83,61.99,20.25,0.56,0.22,9.09,0.28,0.42
9,citroen,40696,9.95,213.66,157.24,0.08,0.95,18.40,0.08,0.52
50,opel,105375,6.13,64.60,17.65,0.04,0.14,6.97,0.03,0.21


In [4]:
df_28_9= pd.read_excel('../data/raw/fz28_2026_09.xlsx', header=None, sheet_name="FZ 28.9")

regions = df_28_9.iloc[31:48, 1]
total_registrations_bundesland = to_count(df_28_9.iloc[31:48, 2])
electric_drive_registrations_bundesland = to_count(df_28_9.iloc[31:48, 5])
bev_registrations_bundesland = to_count(df_28_9.iloc[31:48, 7])
fcev_registrations_bundesland = to_count(df_28_9.iloc[31:48, 8])
phev_registrations_bundesland = to_count(df_28_9.iloc[31:48, 9])

assert df_28_9.iloc[30, 1].strip().lower() == 'januar - september 2026'
assert total_registrations_bundesland.sum() == df_28_9.iloc[30,2]
assert electric_drive_registrations_bundesland.sum() == df_28_9.iloc[30,5]
assert bev_registrations_bundesland.sum() == df_28_9.iloc[30,7]
assert phev_registrations_bundesland.sum() == df_28_9.iloc[30,9]

assert (electric_drive_registrations_bundesland ==
           bev_registrations_bundesland + phev_registrations_bundesland + fcev_registrations_bundesland).all()

regional = pd.DataFrame({
    'bundesland': regions.str.strip(),
    'total_ytd26': total_registrations_bundesland,
    'electric_ytd26': electric_drive_registrations_bundesland,
    'bev_ytd26': bev_registrations_bundesland,
    'phev_ytd26': phev_registrations_bundesland
}).reset_index(drop=True)

regional = regional[regional['bundesland'] != 'Sonstige'].reset_index(drop=True)
assert len(regional) == 16

regional['bev_share_ytd26'] = regional['bev_ytd26'] / regional['total_ytd26'] * 100
regional['phev_share_ytd26'] = regional['phev_ytd26'] / regional['total_ytd26'] * 100
regional['bev_phev_ratio_ytd26'] = regional['bev_ytd26'] / regional['phev_ytd26']

national_total = df_28_9.iloc[30, 2]

national_bev_share = df_28_9.iloc[30,7] / national_total * 100
national_phev_share = df_28_9.iloc[30,9] / national_total * 100
national_bev_phev_ratio = df_28_9.iloc[30,7] / df_28_9.iloc[30,9]
print(national_bev_share, national_phev_share)

regional['bev_share_vs_national_pp'] = regional['bev_share_ytd26'] - national_bev_share
regional['phev_share_vs_national_pp'] = regional['phev_share_ytd26'] - national_phev_share
regional['bev_phev_ratio_vs_national'] = regional['bev_phev_ratio_ytd26'] - national_bev_phev_ratio

display(regional.sort_values(by='bev_share_ytd26', ascending=False).round(2))

27.191670889972897 11.139131460195753


,bundesland,total_ytd26,electric_ytd26,bev_ytd26,phev_ytd26,bev_share_ytd26,phev_share_ytd26,bev_phev_ratio_ytd26,bev_share_vs_national_pp,phev_share_vs_national_pp,bev_phev_ratio_vs_national
10,Rheinland-Pfalz,85556,36699,28286,8413,33.06,9.83,3.36,5.87,-1.31,0.92
14,Schleswig-Holstein,59787,25289,19503,5786,32.62,9.68,3.37,5.43,-1.46,0.93
8,Niedersachsen,246132,100103,73285,26818,29.77,10.90,2.73,2.58,-0.24,0.29
0,Baden-Württemberg,305951,131655,90905,40749,29.71,13.32,2.23,2.52,2.18,-0.21
9,Nordrhein-Westfalen,443825,179349,130092,49256,29.31,11.10,2.64,2.12,-0.04,0.20
4,Bremen,12018,5069,3470,1599,28.87,13.31,2.17,1.68,2.17,-0.27
11,Saarland,23107,8574,6412,2161,27.75,9.35,2.97,0.56,-1.79,0.53
2,Berlin,41345,17120,11445,5673,27.68,13.72,2.02,0.49,2.58,-0.42
3,Brandenburg,37048,13559,9969,3590,26.91,9.69,2.78,-0.28,-1.45,0.34
1,Bayern,467237,170311,124289,46021,26.60,9.85,2.70,-0.59,-1.29,0.26


In [5]:
charging_stations = pd.read_parquet('../data/processed/charging_stations_clean.parquet')

In [6]:
print(charging_stations.shape)
print(charging_stations.columns.tolist())

(61793, 31)
['Betreiber', 'Art der Ladeeinrichung', 'Anzahl Ladepunkte', 'Steckertypen1', 'Steckertypen2', 'Steckertypen3', 'Steckertypen4', 'P1 [kW]', 'P2 [kW]', 'P3 [kW]', 'P4 [kW]', 'Nennleistung Ladeeinrichtung [kW]', 'Kreis/kreisfreie Stadt', 'Ort', 'Postleitzahl', 'Straße', 'Hausnummer', 'Adresszusatz', 'Inbetriebnahmedatum', 'koordinaten', 'Anzeigename (Karte)', 'Steckertypen5', 'P5 [kW]', 'Public Key5', 'Steckertypen6', 'P6 [kW]', 'Public Key6', 'Postleitzahl_original', 'Bundesland', 'has_AC', 'has_DC']


In [7]:
charging_by_bundesland = (
    charging_stations
    .groupby('Bundesland', as_index=False)[['has_AC', 'has_DC']]
    .sum()
)
charging_by_bundesland

,Bundesland,has_AC,has_DC
0,Baden-Württemberg,9509,1763
1,Bayern,10519,2496
2,Berlin,2475,345
3,Brandenburg,1123,425
4,Bremen,384,77
5,Hamburg,1109,238
6,Hessen,3852,926
7,Mecklenburg-Vorpommern,743,223
8,Niedersachsen,5237,1562
9,Nordrhein-Westfalen,10411,2579


In [8]:
ports_by_bundesland = (
    charging_stations
    .groupby('Bundesland', as_index=False)['Anzahl Ladepunkte']
    .sum()
    .rename(columns={'Anzahl Ladepunkte': 'charging_ports'})
)

ports_by_bundesland['charging_ports'] = ports_by_bundesland['charging_ports'].astype(int)

print(pd.concat([charging_stations[f'Steckertypen{i}'] for i in range(1, 7)]).value_counts().head(20))
print(charging_stations['Inbetriebnahmedatum'].min(), charging_stations['Inbetriebnahmedatum'].max())

AC Typ 2 Steckdose                                                                      78837
DC Fahrzeugkupplung Typ Combo 2 (CCS)                                                   19241
AC Typ 2 Fahrzeugkupplung                                                                9396
AC Typ 2 Steckdose; AC Schuko                                                            4783
DC Fahrzeugkupplung Typ Combo 2 (CCS); DC CHAdeMO                                        2571
AC Typ 2 Steckdose; AC Typ 2 Fahrzeugkupplung                                            2012
AC Typ 2 Fahrzeugkupplung; DC Fahrzeugkupplung Typ Combo 2 (CCS)                          338
AC Typ 2 Steckdose; AC Typ 2 Steckdose                                                    209
AC Typ 2 Fahrzeugkupplung; DC CHAdeMO                                                     204
AC Typ 2 Steckdose; DC Fahrzeugkupplung Typ Combo 2 (CCS)                                 114
AC Typ 2 Fahrzeugkupplung; AC Schuko                        

In [9]:
ports = pd.concat(
    [charging_stations[['Bundesland', f'Steckertypen{i}']]
     .rename(columns={f'Steckertypen{i}': 'connectors'})
     for i in range(1, 7)],
    ignore_index=True
).dropna(subset=['connectors'])

ports['is_dc'] = ports['connectors'].str.contains('DC')
ports['is_ac_only'] = ports['connectors'].str.contains('AC') & ~ports['is_dc']

assert (ports['is_dc'] | ports['is_ac_only']).all()

In [10]:
described = charging_stations[[f'Steckertypen{i}' for i in range(1, 7)]].notna().sum(axis=1)
gap = charging_stations['Anzahl Ladepunkte'] - described

print(len(ports), charging_stations['Anzahl Ladepunkte'].sum())
print(gap.value_counts().sort_index())

118219 118219.0
0.0    61793
Name: count, dtype: int64


In [11]:
port_types_by_bundesland = (
    ports
    .groupby('Bundesland', as_index=False)[['is_dc', 'is_ac_only']]
    .sum()
    .rename(columns={'is_dc': 'dc_ports', 'is_ac_only': 'ac_only_ports'})
)

In [12]:
infrastructure = ports_by_bundesland.merge(
    port_types_by_bundesland,
    on='Bundesland',
    validate='one_to_one'
)

assert len(infrastructure) == 16

infrastructure['undescribed_ports'] = (
    infrastructure['charging_ports'] - infrastructure['dc_ports'] - infrastructure['ac_only_ports']
)
infrastructure['dc_port_share'] = (
    infrastructure['dc_ports'] / (infrastructure['dc_ports'] + infrastructure['ac_only_ports']) * 100
)

display(infrastructure.round(2))

,Bundesland,charging_ports,dc_ports,ac_only_ports,undescribed_ports,dc_port_share
0,Baden-Württemberg,20653,2857,17796,0,13.83
1,Bayern,22500,4298,18202,0,19.10
2,Berlin,4432,589,3843,0,13.29
3,Brandenburg,2661,700,1961,0,26.31
4,Bremen,854,116,738,0,13.58
5,Hamburg,2554,443,2111,0,17.35
6,Hessen,8660,1557,7103,0,17.98
7,Mecklenburg-Vorpommern,1643,364,1279,0,22.15
8,Niedersachsen,11828,2622,9206,0,22.17
9,Nordrhein-Westfalen,22882,4416,18466,0,19.30


In [13]:
print(charging_stations['Inbetriebnahmedatum'].dtype)
print(charging_stations['Inbetriebnahmedatum'].notna().sum())

float64
0


In [14]:
population = pd.read_excel('../data/raw/12411-0021_de.xlsx', header=None)

name_cols = list(range(1, 47, 3))
assert (population.iloc[4, [c + 2 for c in name_cols]] == 'Insgesamt').all()

population_final = pd.DataFrame({
    'bundesland': [population.iloc[3, c].strip() for c in name_cols],
    'population': pd.to_numeric(population.iloc[24, [c + 2 for c in name_cols]]).to_numpy()
})

population_final.sort_values(by='population',ascending=False)
print(population.iloc[3:6, 0:7])
print(population.iloc[24, 0])

                           0                  1         2          3         4         5          6
3  Stichtag zum Quartalsende  Baden-Württemberg       NaN        NaN    Bayern       NaN        NaN
4                        NaN           männlich  weiblich  Insgesamt  männlich  weiblich  Insgesamt
5                 30.09.2021            5525810   5597583   11123393   6532463   6644181   13176644
30.06.2026


/Users/cankale/miniforge3/envs/auto_env/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [15]:
assert len(population_final) == 16
assert set(population_final['bundesland']) == set(regional['bundesland'])
print(population_final['population'].sum())

83298787


In [16]:
population_final.to_parquet('../data/processed/population_by_bundesland.parquet', index=False)

In [44]:
area = pd.read_excel('../data/raw/11111-0001_de.xlsx')

rows = list(range(4, 20))

area_final = pd.DataFrame({
    'bundesland': [area.iloc[r, 0].strip() for r in rows],
    'area_in_sqkm': pd.to_numeric(area.iloc[rows , 1]).to_numpy()
})


insgesamt = area.iloc[20, 1]

assert np.isclose(area_final['area_in_sqkm'].sum(), insgesamt)

print(area.iloc[20, 0])
assert len(area_final) == 16
assert set(area_final['bundesland']) == set(regional['bundesland'])

Insgesamt


/Users/cankale/miniforge3/envs/auto_env/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
